# DeepConvLSTM Replication v2 — Fixed Pipeline

This notebook is a corrected replication of `DeepConv+LSTM.ipynb`.

## Issues fixed vs v1 (`replication_deepconvlstm.ipynb`)

| Issue | v1 (broken) | v2 (this notebook) |
|---|---|---|
| Training | `FORCE_RETRAIN=False` → 0 epochs ran | Always trains from scratch |
| Normalization | Train-only `StandardScaler` (data leakage-free but different scale) | **Global** `StandardScaler` fit on full dataset **before** split — matches original |
| Windowing | Per-user, drops cross-boundary windows | Naive sliding window over full sorted DataFrame — matches original |
| Split | 3-way train / val / test | 2-way `train_test_split(test_size=0.33, random_state=42)` — matches original |
| PTQ calibration | 256 samples from train (`traincal`) | 100 samples from `x_test` — matches original |

## Target results (from original notebook)
- Baseline accuracy: **98.24 %**
- PTQ (int8) accuracy: **97.09 %**
- PTQ model size: **136.51 KB**

In [1]:
from pathlib import Path
import sys
import os
import importlib.util
import warnings
warnings.filterwarnings('ignore')

# ── GPU / CUDA / XLA runtime guardrails (must run before TF import) ──────────
# Auto-detects GPU availability; override manually if needed:
#   USE_GPU = True   — force GPU (fails if no CUDA runtime found)
#   USE_GPU = False  — force CPU (works on macOS / any machine without an NVIDIA GPU)
import platform as _platform
_has_nvidia_dir = (Path(sys.prefix) / f'lib/python{sys.version_info[0]}.{sys.version_info[1]}/site-packages/nvidia').exists()
USE_GPU = _has_nvidia_dir and _platform.system() != 'Darwin'

if not USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

py_major, py_minor = sys.version_info[:2]
nvidia_root    = Path(sys.prefix) / f'lib/python{py_major}.{py_minor}/site-packages/nvidia'
cuda_nvcc_root = nvidia_root / 'cuda_nvcc'
ptxas_bin_dir  = cuda_nvcc_root / 'bin'
ptxas_path     = ptxas_bin_dir / 'ptxas'

if USE_GPU and nvidia_root.exists():
    # Ensure CUDA shared libraries (libnvrtc etc.) are discoverable
    lib_dirs = [p for p in sorted(nvidia_root.glob('*/lib')) if p.is_dir()]
    existing_ld = os.environ.get('LD_LIBRARY_PATH', '')
    ld_entries = existing_ld.split(':') if existing_ld else []
    for lib_dir in reversed(lib_dirs):
        s = str(lib_dir)
        if s not in ld_entries:
            ld_entries.insert(0, s)
    os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_entries)

    # Ensure ptxas is on PATH for XLA CUDA compilation
    if ptxas_path.exists():
        path_entries = os.environ.get('PATH', '').split(':')
        if str(ptxas_bin_dir) not in path_entries:
            os.environ['PATH'] = f'{ptxas_bin_dir}:{os.environ.get("PATH", "")}'.strip(':')

        xla_flag = f'--xla_gpu_cuda_data_dir={cuda_nvcc_root}'
        existing_xla = os.environ.get('XLA_FLAGS', '')
        if xla_flag not in existing_xla:
            os.environ['XLA_FLAGS'] = (existing_xla + ' ' + xla_flag).strip()

# ── Check required packages ───────────────────────────────────────────────────
required_modules = {
    'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
    'seaborn': 'seaborn', 'scipy': 'scipy', 'sklearn': 'scikit-learn',
    'tensorflow': 'tensorflow',
}
missing = [f'{mod} (pip: {pkg})' for mod, pkg in required_modules.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    raise ModuleNotFoundError('Missing dependencies:\n- ' + '\n- '.join(missing))

# ── Standard imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    precision_recall_fscore_support, classification_report,
)

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from IPython.display import display

# ── TF GPU memory growth ──────────────────────────────────────────────────────
if USE_GPU:
    for g in tf.config.list_physical_devices('GPU'):
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception as e:
            print(f'[WARN] Memory growth: {e}')

# ── Repo root on path ─────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ── src/ imports — used directly, no modifications ───────────────────────────
from src.models.deepconv_lstm import build_deepconv_lstm, compile_deepconv_lstm
from src.data.load_wisdm import load_wisdm_dataframe
from src.data.preprocess_zhou2025 import preprocess_zhou2025
from src.utils.repro import set_global_seed
from src.utils.runtime import check_tensorflow_runtime

# ── libnvrtc check (fail-fast if GPU mode but runtime missing) ────────────────
nvrtc_candidates = []
if nvidia_root.exists():
    for lib_dir in sorted(nvidia_root.glob('*/lib')):
        nvrtc_candidates.extend(sorted(lib_dir.glob('libnvrtc.so*')))
if USE_GPU and not nvrtc_candidates:
    raise RuntimeError(
        'GPU mode requested but libnvrtc.so not found. '
        'Run: conda env update -n tinymlproj -f environment.yml --prune'
    )

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = REPO_ROOT / 'WISDM_ar_v1.1'
CSV_PATH = DATA_DIR / 'WISDM_ar_v1.1_raw.csv'

SEED = 42
set_global_seed(SEED)

print(f'Repo root:  {REPO_ROOT}')
print(f'USE_GPU:    {USE_GPU}')
print(f'TF version: {tf.__version__}')
print(f'Visible GPUs: {tf.config.list_physical_devices("GPU")}')
print(f'nvidia_root exists: {nvidia_root.exists()}')
print(f'ptxas found: {ptxas_path.exists()}')
print(f'libnvrtc candidates: {[str(p) for p in nvrtc_candidates[:3]]}')
print(f'XLA_FLAGS: {os.environ.get("XLA_FLAGS", "")}')
assert CSV_PATH.exists(), f'Dataset not found: {CSV_PATH}'

2026-03-02 07:51:23.715319: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Repo root:  /Users/hamza/Documents/GitHub/har-mcu
USE_GPU:    False
TF version: 2.14.1
Visible GPUs: []
nvidia_root exists: False
ptxas found: False
libnvrtc candidates: []
XLA_FLAGS: 


In [2]:
# src/utils/runtime.py — check_tensorflow_runtime — used directly, no modifications
runtime_status = check_tensorflow_runtime({'env': {'require_gpu': USE_GPU}})
display(pd.DataFrame([runtime_status]))

if 'error' in runtime_status:
    raise RuntimeError(runtime_status['error'])

,tensorflow_ok,version_ok,gpu_ok,tensorflow_version,gpus
0,True,True,False,2.14.1,[]


## 1. Load and clean data

Uses **`src/data/load_wisdm.py`** and **`src/data/preprocess_zhou2025.py`** directly — no modifications.  
Steps: CSV read → column sanity check → dropna → drop zero-timestamps → sort by (user, timestamp).

In [3]:
# src/data/load_wisdm.py — plain CSV read + column/sanity check
minimal_cfg = {"paths": {"raw_csv": str(CSV_PATH)}}
raw_df, sanity = load_wisdm_dataframe(minimal_cfg)
print('Sanity:', sanity)

# src/data/preprocess_zhou2025.py — dropna + drop zero-ts + sort by (user, timestamp)
df, pre_stats = preprocess_zhou2025(raw_df)
print('Pre-process stats:', pre_stats)

print('\nShape after cleaning:', df.shape)
print('Unique users:', df['user'].nunique())
print()
print('Class distribution:')
print(df['activity'].value_counts())

Sanity: {'rows': 1073623, 'missing_values': 0, 'zero_timestamps': 0}
Pre-process stats: {'rows_before': 1073623, 'rows_after_dropna': 1073623, 'rows_after_drop_zero_timestamp': 1073623, 'rows_final': 1073623, 'dropped_total': 0}

Shape after cleaning: (1073623, 6)
Unique users: 36

Class distribution:
activity
Walking       417901
Jogging       324600
Upstairs      122598
Downstairs    100192
Sitting        59939
Standing       48393
Name: count, dtype: int64


## 2. Label encoding + Global normalization

**Key fix:** `StandardScaler` is fit on the **entire dataset** before any train/test split.  
This is exactly what the original notebook did and is the primary reason it achieved 98.24 % accuracy.  
(v1 fit the scaler only on training windows, producing different input statistics at inference.)

In [4]:
# ── Label encoding ───────────────────────────────────────────────────────────
LABEL = 'ActivityEncoded'
le = preprocessing.LabelEncoder()
df[LABEL] = le.fit_transform(df['activity'].values.ravel())

LABELS = list(le.classes_)
print('Classes:', LABELS)
print('Encoded:', le.transform(LABELS))

# ── Global StandardScaler — fit on FULL dataset before split ─────────────────
axis_cols = ['x-axis', 'y-axis', 'z-axis']
scaler = StandardScaler()
df[axis_cols] = scaler.fit_transform(df[axis_cols])

print('\nScaler mean:', scaler.mean_.round(4))
print('Scaler std: ', scaler.scale_.round(4))
print('\nSample after scaling:')
display(df[axis_cols + ['activity', LABEL]].head(3))

Classes: ['Downstairs', 'Jogging', 'Sitting', 'Standing', 'Upstairs', 'Walking']
Encoded: [0 1 2 3 4 5]

Scaler mean: [0.6721 7.3327 0.4025]
Scaler std:  [6.9169 6.7329 4.7893]

Sample after scaling:


,x-axis,y-axis,z-axis,activity,ActivityEncoded
0,0.002594,0.514975,-0.507894,Walking,5
1,0.893168,0.015937,-0.188432,Walking,5
2,0.037292,-0.252891,-0.188432,Walking,5


## 3. Windowing

**Key fix:** Naive sliding window over the full sorted DataFrame with no per-user boundary enforcement.  
The original iterates `range(0, len(df) - time_steps, step)` — windows can span across user boundaries.  
(v1 windowed per-user and dropped cross-boundary windows, reducing the effective dataset size.)

In [5]:
TIME_PERIODS  = 100   # window size (matches original)
STEP_DISTANCE = 50    # 50 % overlap (matches original)
N_FEATURES    = 3     # x, y, z

def create_segments_and_labels(df, time_steps, step, label_name):
    """Naive sliding window — identical to the original notebook."""
    segments = []
    labels   = []
    for i in range(0, len(df) - time_steps, step):
        xs = df['x-axis'].values[i: i + time_steps]
        ys = df['y-axis'].values[i: i + time_steps]
        zs = df['z-axis'].values[i: i + time_steps]
        label = stats.mode(df[label_name][i: i + time_steps], keepdims=True)[0][0]
        segments.append([xs, ys, zs])
        labels.append(label)

    X = np.asarray(segments, dtype=np.float32).reshape(-1, time_steps, N_FEATURES)
    y = np.asarray(labels)
    return X, y

X, y = create_segments_and_labels(df, TIME_PERIODS, STEP_DISTANCE, LABEL)

print(f'Windows: {X.shape}   (samples × timesteps × features)')
print(f'Labels:  {y.shape}')

Y_one_hot = to_categorical(y)
print(f'One-hot: {Y_one_hot.shape}')

Windows: (21471, 100, 3)   (samples × timesteps × features)
Labels:  (21471,)
One-hot: (21471, 6)


## 4. Train / test split

**Key fix:** 2-way split (`test_size=0.33, random_state=42`), **no validation set** — matches the original exactly.

In [6]:
x_train, x_test, y_train_one_hot, y_test_one_hot = train_test_split(
    X, Y_one_hot,
    test_size=0.33,
    random_state=42,
)

print(f'Train: {x_train.shape}  |  Test: {x_test.shape}')
print(f'Train labels: {y_train_one_hot.shape}  |  Test labels: {y_test_one_hot.shape}')

Train: (14385, 100, 3)  |  Test: (7086, 100, 3)
Train labels: (14385, 6)  |  Test labels: (7086, 6)


## 5. Build model

Uses **`src/models/deepconv_lstm.py`** directly — no modifications.  
Architecture: Conv1D(32) → Dropout → Conv1D(64) → Dropout → LSTM(100) → Flatten → Dropout → Dense(6, softmax).

In [7]:
# src/models/deepconv_lstm.py — build_deepconv_lstm + compile_deepconv_lstm
# Used directly, no modifications.
model = build_deepconv_lstm(
    window_size=TIME_PERIODS,
    num_features=N_FEATURES,
    num_classes=len(LABELS),
    dropout=0.3,
)
model = compile_deepconv_lstm(model)   # RMSprop lr=0.001, categorical_crossentropy

model.summary()

Model: "deepconv_lstm"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1 (Conv1D)              (None, 98, 32)            320       
                                                                 
 dropout1 (Dropout)          (None, 98, 32)            0         
                                                                 
 conv2 (Conv1D)              (None, 96, 64)            6208      
                                                                 
 dropout2 (Dropout)          (None, 96, 64)            0         
                                                                 
 lstm (LSTM)                 (None, 96, 100)           66000     
                                                                 
 flatten (Flatten)           (None, 9600)              0         
                                                                 
 dropout3 (Dropout)          (None, 9600)            

## 6. Train

Always trains from scratch (no checkpoint reuse). Uses `x_test` as validation data during training to track the val curve — identical to the original.

In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
]

history = model.fit(
    x_train, y_train_one_hot,
    epochs=50,
    batch_size=64,
    validation_data=(x_test, y_test_one_hot),
    callbacks=callbacks,
)

# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['accuracy'],     label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'],     label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Save checkpoint
ckpt_dir = REPO_ROOT / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / 'deepconv_lstm_v2.weights.h5'
model.save_weights(str(ckpt_path))
print('Saved weights to', ckpt_path)

Epoch 1/50
225/225 [==============================] - 115s 487ms/step - loss: 0.6529 - accuracy: 0.7615 - val_loss: 0.4599 - val_accuracy: 0.8408 - lr: 0.0010
Epoch 2/50
225/225 [==============================] - 51s 226ms/step - loss: 0.4515 - accuracy: 0.8385 - val_loss: 0.3367 - val_accuracy: 0.8851 - lr: 0.0010
Epoch 3/50
225/225 [==============================] - 43s 193ms/step - loss: 0.3520 - accuracy: 0.8692 - val_loss: 0.3074 - val_accuracy: 0.8857 - lr: 0.0010
Epoch 4/50
225/225 [==============================] - 43s 190ms/step - loss: 0.2972 - accuracy: 0.8893 - val_loss: 0.2483 - val_accuracy: 0.9101 - lr: 0.0010
Epoch 5/50
225/225 [==============================] - 46s 205ms/step - loss: 0.2563 - accuracy: 0.9077 - val_loss: 0.2448 - val_accuracy: 0.9080 - lr: 0.0010
Epoch 6/50
225/225 [==============================] - 44s 196ms/step - loss: 0.2227 - accuracy: 0.9196 - val_loss: 0.2111 - val_accuracy: 0.9220 - lr: 0.0010
Epoch 7/50
225/225 [==============================]

## 7. Baseline (float) evaluation

In [ ]:
loss, float_accuracy = model.evaluate(x_test, y_test_one_hot, verbose=0)
print(f'Test Loss:     {loss:.4f}')
print(f'Test Accuracy: {float_accuracy:.4f}   (target: 0.9824)')

y_pred_prob = model.predict(x_test)
y_pred = y_pred_prob.argmax(axis=-1)
y_true = y_test_one_hot.argmax(axis=-1)

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
print(f'Precision: {precision:.4f}  Recall: {recall:.4f}  F1: {f1:.4f}')

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=LABELS))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(
    pd.DataFrame(cm, index=LABELS, columns=LABELS),
    annot=True, fmt='d', cmap='Blues', linewidths=0.5,
    cbar=False, annot_kws={'size': 12}, square=True,
)
plt.title(f'Baseline (Float)  —  Accuracy: {float_accuracy:.4f}')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

## 8. Post-Training Quantization (PTQ int8)

**Key fix:** Calibration uses **100 samples from `x_test`** — matches the original notebook's representative dataset construction.

In [ ]:
# Fix input shape to batch=1 for conversion
model.input.set_shape((1,) + model.input.shape[1:])

def representative_dataset_gen():
    """100 samples from x_test — matches original authorcal approach."""
    for sample in tf.data.Dataset.from_tensor_slices(x_test).batch(1).take(100):
        yield [tf.cast(sample, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_model_quant = converter.convert()

# Save
ptq_path = REPO_ROOT / 'models_tflite' / 'deepconv_lstm_v2_ptq_int8.tflite'
ptq_path.parent.mkdir(parents=True, exist_ok=True)
ptq_path.write_bytes(tflite_model_quant)

model_size_kb = ptq_path.stat().st_size / 1024
print(f'PTQ int8 model saved to: {ptq_path}')
print(f'Model size: {model_size_kb:.2f} KB   (target: 136.51 KB)')

# Report I/O dtypes
interp_check = tf.lite.Interpreter(model_content=tflite_model_quant)
print(f'Input  dtype: {interp_check.get_input_details()[0]["dtype"]}')
print(f'Output dtype: {interp_check.get_output_details()[0]["dtype"]}')

## 9. Evaluate PTQ model

In [ ]:
interp = tf.lite.Interpreter(model_path=str(ptq_path))
interp.allocate_tensors()

input_det  = interp.get_input_details()[0]
output_det = interp.get_output_details()[0]
in_scale, in_zero = input_det['quantization']

def predict_tflite(x_single):
    q = (x_single / in_scale + in_zero).astype(np.int8)
    interp.set_tensor(input_det['index'], q)
    interp.invoke()
    return interp.get_tensor(output_det['index'])

y_true_ptq = []
y_pred_ptq = []
for i in range(len(x_test)):
    out = predict_tflite(x_test[i:i+1])
    y_pred_ptq.append(int(np.argmax(out)))
    y_true_ptq.append(int(np.argmax(y_test_one_hot[i])))

ptq_accuracy = accuracy_score(y_true_ptq, y_pred_ptq)
print(f'PTQ int8 accuracy: {ptq_accuracy:.4f}   (target: 0.9709)')

print('\nClassification report (PTQ):')
print(classification_report(y_true_ptq, y_pred_ptq, target_names=LABELS))

cm_ptq = confusion_matrix(y_true_ptq, y_pred_ptq)
plt.figure(figsize=(10, 8))
sns.heatmap(
    pd.DataFrame(cm_ptq, index=LABELS, columns=LABELS),
    annot=True, fmt='d', cmap='Oranges', linewidths=0.5,
    cbar=False, annot_kws={'size': 12}, square=True,
)
plt.title(f'PTQ int8 Model  —  Accuracy: {ptq_accuracy:.4f}')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

## 10. Summary / Verdict

In [ ]:
TARGET_BASELINE_ACC = 0.9824
TARGET_PTQ_ACC      = 0.9709
TARGET_PTQ_SIZE_KB  = 136.51

TOL_ACC  = 0.02   # ±2 pp tolerance
TOL_SIZE = 20.0   # ±20 KB tolerance

def verdict(value, target, tol):
    return '✅ PASS' if abs(value - target) <= tol else '⚠️  WARN'

summary = pd.DataFrame([
    {
        'metric':   'Baseline accuracy',
        'target':   f'{TARGET_BASELINE_ACC:.4f}',
        'achieved': f'{float_accuracy:.4f}',
        'delta':    f'{float_accuracy - TARGET_BASELINE_ACC:+.4f}',
        'status':   verdict(float_accuracy, TARGET_BASELINE_ACC, TOL_ACC),
    },
    {
        'metric':   'PTQ int8 accuracy',
        'target':   f'{TARGET_PTQ_ACC:.4f}',
        'achieved': f'{ptq_accuracy:.4f}',
        'delta':    f'{ptq_accuracy - TARGET_PTQ_ACC:+.4f}',
        'status':   verdict(ptq_accuracy, TARGET_PTQ_ACC, TOL_ACC),
    },
    {
        'metric':   'PTQ model size (KB)',
        'target':   f'{TARGET_PTQ_SIZE_KB:.2f}',
        'achieved': f'{model_size_kb:.2f}',
        'delta':    f'{model_size_kb - TARGET_PTQ_SIZE_KB:+.2f}',
        'status':   verdict(model_size_kb, TARGET_PTQ_SIZE_KB, TOL_SIZE),
    },
])

display(summary)

## Notes on Methodology

This notebook deliberately reproduces the results of the original `DeepConv+LSTM.ipynb` as closely as possible, including intentionally replicating its methodology — flaws and all. The goal is faithful numeric replication, not a clean production pipeline.

The original methodology contains several well-known issues:

### 1. Global normalization (data leakage)
`StandardScaler` is fit on the **entire dataset** before the train/test split. This means the scaler has seen the test samples' statistics during fitting, which constitutes data leakage. A correct pipeline would fit the scaler only on the training set and apply it to the test set.

### 2. No validation set (test set used during training)
The original uses a 2-way split — there is no separate validation set. The `x_test` partition is passed directly as `validation_data` to `model.fit()`. This means the `EarlyStopping` and `ReduceLROnPlateau` callbacks, which monitor `val_loss`, are making training decisions based on the held-out test data. The model effectively "sees" the test set during training, which inflates the reported test accuracy.

### 3. Naive sliding window with no user boundary enforcement
The windowing passes a single sliding window over the full sorted DataFrame without resetting at user boundaries. Windows can therefore span two different users' recordings, mixing their sensor streams into the same training example. A correct approach would window per-user and discard any window that crosses a session or user boundary.

### 4. No reproducibility of the original split
The original notebook does not save or hash the split, making exact numerical reproduction across environments sensitive to library versions and floating-point ordering.

---

These issues are documented here for transparency. `replication_deepconvlstm.ipynb` (v1) addresses points 1–3 with a proper train-only scaler, a 3-way split, and per-user windowing — at the cost of lower reported accuracy, which better reflects true generalisation performance.